# Shear Check Comparison: CircularSectionCheck vs ShearCheck

This notebook compares `CircularSectionCheck.perform_shear_check` (Orr 2012) against
the standard `ShearCheck.perform_check` when applied to the **same** circular section.

**Key differences investigated:**
- Equivalent web width `b_w` (circular chord vs polygon minimum)
- Strut angle `cot(θ)` (driven by different `K` values)
- Reinforcement capacity `V_Rd,s` (with λ₁/λ₂ efficiency factors)
- Strut crushing capacity `V_Rd,max`
- Uncracked vs cracked behaviour and `force_cracked` effect

In [1]:
import warnings
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from materials.reinforced_concrete.geometry import (
    create_circular_section,
    create_circular_perimeter_rebars,
)
from materials.reinforced_concrete.materials import (
    ConcreteMaterial,
    Rebar,
    ShearRebar,
)
from materials.reinforced_concrete.code_checks.ec2_2004 import (
    CircularSectionCheck,
    ShearCheck,
    ShearLoadCase,
)

warnings.filterwarnings("ignore")

---
## 1. Common Section Setup

Both checks use the identical section, concrete, and shear reinforcement.

In [2]:
# Section parameters
D = 600       # Diameter (mm)
cover = 50    # Cover to link outer face (mm)
link_dia = 12
bar_dia = 20
n_bars = 10

# Build section
section = create_circular_section(diameter=D, section_name=f"{D}mm Pile")
rebar = Rebar(diameter=bar_dia)
rebar_cover = cover + link_dia + bar_dia / 2
rebar_group = create_circular_perimeter_rebars(
    rebar=rebar, diameter=D, cover=rebar_cover,
    n_bars=n_bars, origin=(D / 2, D / 2),
)
section.add_rebar_group(rebar_group)

# Materials
concrete = ConcreteMaterial(grade="C30/37")
links = ShearRebar(diameter=link_dia, spacing=200, n_legs=2, grade="B500B")

# Create both checks
circ = CircularSectionCheck(
    section=section, concrete=concrete, diameter=D,
    cover=cover, shear_reinforcement=links,
)

std = ShearCheck(
    section=section, concrete=concrete,
    shear_reinforcement=links,
    use_rigorous=True, cap_lever_arm=True,
)

print(f"Section: {D}mm circular, {n_bars}H{bar_dia}, H{link_dia}@{links.spacing}")
print(f"Concrete: {concrete.grade} (f_ck={concrete.f_ck} MPa, f_cd={concrete.f_cd:.2f} MPa)")
print(f"")
print(f"ShearCheck b_w:       {std.breadth:.1f} mm  (polygon minimum width)")
print(f"CircularSC r_sv:      {circ.r_sv:.1f} mm")
print(f"\u03bb\u2081 (numerical):       {circ.calculate_lambda_1(100, 350):.4f}")
print(f"\u03bb\u2082 (closed links):    {circ.calculate_lambda_2():.4f}")

Section: 600mm circular, 10H20, H12@200.0
Concrete: C30/37 (f_ck=30.0 MPa, f_cd=20.00 MPa)

ShearCheck b_w:       165.4 mm  (polygon minimum width)
CircularSC r_sv:      244.0 mm
λ₁ (numerical):       0.8250
λ₂ (closed links):    1.0000


---
## 2. Side-by-Side Comparison: Single Load Case

Detailed breakdown of every intermediate value for a common load case.

In [3]:
lc = ShearLoadCase(V_Ed=250, M_Ed=150, N_Ed=500)

r_std = std.perform_check(load_case=lc)
r_circ = circ.perform_shear_check(load_case=lc, force_cracked=True)

ds = r_std.details
dc = r_circ.details

print(f"Load: V_Ed={lc.V_Ed} kN, M_Ed={lc.M_Ed} kN\u00b7m, N_Ed={lc.N_Ed} kN")
print()
print(f"{'Parameter':<28} | {'ShearCheck':>12} | {'CircularSC':>12} | {'Ratio':>8}")
print("-" * 68)

rows = [
    ("d (mm)", ds['d'], dc['d']),
    ("z (mm)", ds['z'], dc['z']),
    ("b_w (mm)", ds['b_w'], dc['b_w']),
    ("\u03c3_cp (MPa)", ds['sigma_cp'], dc['sigma_cp']),
    ("\u03b1_cw", ds.get('alpha_cw', '-'), dc.get('alpha_cw', '-')),
    ("\u03bd\u2081", ds.get('nu_1', '-'), dc.get('nu_1', '-')),
    ("cot(\u03b8)", ds['cot_theta'], dc['cot_theta']),
    ("\u03b8 (deg)", ds['theta_deg'], dc['theta_deg']),
    ("\u03bb\u2081", "-", dc.get('lambda_1', '-')),
    ("\u03bb\u2082", "-", dc.get('lambda_2', '-')),
    ("V_Rd,s (kN)", ds['V_Rd_s'], dc['V_Rd_s']),
    ("V_Rd,max (kN)", ds['V_Rd_max'], dc['V_Rd_max']),
    ("V_Rd (kN)", ds['V_Rd'], dc['V_Rd']),
    ("Utilization", r_std.utilization, r_circ.utilization),
]

for name, v_std, v_circ in rows:
    if isinstance(v_std, str) or isinstance(v_circ, str):
        print(f"{name:<28} | {str(v_std):>12} | {str(v_circ):>12} |      -")
    else:
        ratio = v_circ / v_std if v_std != 0 else float('inf')
        if name == "Utilization":
            print(f"{name:<28} | {v_std:>11.1%} | {v_circ:>11.1%} | {ratio:>7.2f}")
        else:
            print(f"{name:<28} | {v_std:>12.1f} | {v_circ:>12.1f} | {ratio:>7.2f}")

Load: V_Ed=250.0 kN, M_Ed=150.0 kN·m, N_Ed=500.0 kN

Parameter                    |   ShearCheck |   CircularSC |    Ratio
--------------------------------------------------------------------
d (mm)                       |        411.8 |        411.8 |    1.00
z (mm)                       |        348.6 |        348.6 |    1.00
b_w (mm)                     |        165.4 |        368.4 |    2.23
σ_cp (MPa)                   |          1.7 |          1.8 |    1.06
α_cw                         |          1.1 |          1.1 |    1.00
ν₁                           |          0.5 |          0.5 |    1.00
cot(θ)                       |          2.2 |          2.5 |    1.15
θ (deg)                      |         24.6 |         21.8 |    0.89
λ₁                           |            - | 0.855553366578033 |      -
λ₂                           |            - |          1.0 |      -
V_Rd,s (kN)                  |        373.8 |        366.6 |    0.98
V_Rd,max (kN)                |        329.9 | 

---
## 3. cot(θ) Comparison Across Shear Forces

The strut angle is driven by `V_Rd,max = K / (cotθ + tanθ)`. Since `K` depends
on `b_w`, the different web widths produce different strut angles for the same V_Ed.

- **ShearCheck**: uses polygon minimum width (very narrow for circles)
- **CircularSectionCheck**: uses Orr equivalent `b_w = min(b_wc, b_wt)`

In [4]:
M_fixed, N_fixed = 150, 500
V_range = np.arange(50, 750, 25)

cot_std_list = []
cot_circ_list = []

for V in V_range:
    lc = ShearLoadCase(V_Ed=float(V), M_Ed=M_fixed, N_Ed=N_fixed)
    rs = std.perform_check(load_case=lc)
    rc = circ.perform_shear_check(load_case=lc, force_cracked=True)
    cot_std_list.append(rs.details['cot_theta'])
    cot_circ_list.append(rc.details['cot_theta'])

fig = go.Figure()
fig.add_trace(go.Scatter(x=V_range, y=cot_std_list, name="ShearCheck", mode="lines+markers", marker=dict(size=4)))
fig.add_trace(go.Scatter(x=V_range, y=cot_circ_list, name="CircularSectionCheck", mode="lines+markers", marker=dict(size=4)))
fig.add_hline(y=2.5, line_dash="dot", line_color="gray", annotation_text="cot\u03b8 = 2.5 (21.8\u00b0)")
fig.add_hline(y=1.0, line_dash="dot", line_color="gray", annotation_text="cot\u03b8 = 1.0 (45\u00b0)")
fig.update_layout(
    title=f"cot(\u03b8) vs V_Ed (M={M_fixed} kN\u00b7m, N={N_fixed} kN)",
    xaxis_title="V_Ed (kN)", yaxis_title="cot(\u03b8)",
    yaxis=dict(range=[0.8, 2.8]), height=450,
)
fig.show()

---
## 4. Capacity Comparison: V_Rd,s and V_Rd,max

Compare the reinforcement and strut crushing capacities across a range of shear forces.

In [5]:
vrds_std = []; vrdmax_std = []; vrd_std = []
vrds_circ = []; vrdmax_circ = []; vrd_circ = []

for V in V_range:
    lc = ShearLoadCase(V_Ed=float(V), M_Ed=M_fixed, N_Ed=N_fixed)
    rs = std.perform_check(load_case=lc)
    rc = circ.perform_shear_check(load_case=lc, force_cracked=True)

    vrds_std.append(rs.details['V_Rd_s'])
    vrdmax_std.append(rs.details['V_Rd_max'])
    vrd_std.append(min(rs.details['V_Rd_s'], rs.details['V_Rd_max']))

    vrds_circ.append(rc.details['V_Rd_s'])
    vrdmax_circ.append(rc.details['V_Rd_max'])
    vrd_circ.append(min(rc.details['V_Rd_s'], rc.details['V_Rd_max']))

fig = make_subplots(rows=1, cols=2, subplot_titles=("ShearCheck (standard)", "CircularSectionCheck (Orr)"))

# Standard
fig.add_trace(go.Scatter(x=V_range, y=vrds_std, name="V_Rd,s", line=dict(color="blue")), row=1, col=1)
fig.add_trace(go.Scatter(x=V_range, y=vrdmax_std, name="V_Rd,max", line=dict(color="red")), row=1, col=1)
fig.add_trace(go.Scatter(x=V_range, y=list(V_range), name="V_Ed", line=dict(color="black", dash="dash")), row=1, col=1)

# Circular
fig.add_trace(go.Scatter(x=V_range, y=vrds_circ, name="V_Rd,s (circ)", line=dict(color="blue"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=V_range, y=vrdmax_circ, name="V_Rd,max (circ)", line=dict(color="red"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=V_range, y=list(V_range), name="V_Ed", line=dict(color="black", dash="dash"), showlegend=False), row=1, col=2)

fig.update_layout(title="Capacity Breakdown vs V_Ed", height=450)
fig.update_xaxes(title_text="V_Ed (kN)")
fig.update_yaxes(title_text="Capacity (kN)")
fig.show()

---
## 5. Governing Capacity and Utilization

Direct comparison of V_Rd = min(V_Rd,s, V_Rd,max) and utilization.

In [6]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Governing V_Rd", "Utilization"))

fig.add_trace(go.Scatter(x=V_range, y=vrd_std, name="ShearCheck", line=dict(color="blue")), row=1, col=1)
fig.add_trace(go.Scatter(x=V_range, y=vrd_circ, name="CircularSC", line=dict(color="orange")), row=1, col=1)
fig.add_trace(go.Scatter(x=V_range, y=list(V_range), name="V_Ed", line=dict(color="black", dash="dash")), row=1, col=1)

util_std = [v / vr if vr > 0 else float('inf') for v, vr in zip(V_range, vrd_std)]
util_circ = [v / vr if vr > 0 else float('inf') for v, vr in zip(V_range, vrd_circ)]

fig.add_trace(go.Scatter(x=V_range, y=util_std, name="ShearCheck", line=dict(color="blue"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=V_range, y=util_circ, name="CircularSC", line=dict(color="orange"), showlegend=False), row=1, col=2)
fig.add_hline(y=1.0, line_dash="dash", line_color="red", row=1, col=2)

fig.update_xaxes(title_text="V_Ed (kN)")
fig.update_yaxes(title_text="kN", row=1, col=1)
fig.update_yaxes(title_text="Utilization", row=1, col=2)
fig.update_layout(height=400)
fig.show()

---
## 6. Effect of Axial Load on cot(θ)

Axial compression affects σ_cp which influences α_cw and therefore K.

In [7]:
V_fixed, M_fixed = 200, 100
N_range = np.arange(-500, 1800, 100)

cot_std_N = []; cot_circ_N = []
vrds_std_N = []; vrds_circ_N = []

for N in N_range:
    lc = ShearLoadCase(V_Ed=V_fixed, M_Ed=M_fixed, N_Ed=float(N))
    rs = std.perform_check(load_case=lc)
    rc = circ.perform_shear_check(load_case=lc, force_cracked=True)
    cot_std_N.append(rs.details['cot_theta'])
    cot_circ_N.append(rc.details['cot_theta'])
    vrds_std_N.append(rs.details['V_Rd_s'])
    vrds_circ_N.append(rc.details['V_Rd_s'])

fig = make_subplots(rows=1, cols=2, subplot_titles=("cot(\u03b8) vs N_Ed", "V_Rd,s vs N_Ed"))
fig.add_trace(go.Scatter(x=N_range, y=cot_std_N, name="ShearCheck"), row=1, col=1)
fig.add_trace(go.Scatter(x=N_range, y=cot_circ_N, name="CircularSC"), row=1, col=1)
fig.add_trace(go.Scatter(x=N_range, y=vrds_std_N, name="ShearCheck", showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=N_range, y=vrds_circ_N, name="CircularSC", showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="N_Ed (kN)")
fig.update_yaxes(title_text="cot(\u03b8)", row=1, col=1)
fig.update_yaxes(title_text="V_Rd,s (kN)", row=1, col=2)
fig.update_layout(title=f"Effect of Axial Load (V_Ed={V_fixed} kN, M_Ed={M_fixed} kN\u00b7m)", height=400)
fig.show()

---
## 7. Like-for-Like: cot(θ) Override

To isolate the effect of λ₁ and b_w on V_Rd,s (removing the strut angle
difference), we can pass the same `cot_theta_override` to both checks.

In [8]:
cot_values = [1.0, 1.5, 2.0, 2.5]
lc = ShearLoadCase(V_Ed=200, M_Ed=150, N_Ed=500)

print(f"Load: V_Ed={lc.V_Ed} kN, M_Ed={lc.M_Ed} kN\u00b7m, N_Ed={lc.N_Ed} kN")
print()
print(f"{'cot(\u03b8)':<8} | {'V_Rd,s std':>11} | {'V_Rd,s circ':>12} | {'Ratio':>7} | {'V_Rd,max std':>13} | {'V_Rd,max circ':>14}")
print("-" * 80)

for ct in cot_values:
    rs = std.perform_check(load_case=lc, cot_theta_override=ct)
    rc = circ.perform_shear_check(load_case=lc, cot_theta_override=ct, force_cracked=True)

    vs_s = rs.details['V_Rd_s']
    vs_c = rc.details['V_Rd_s']
    vm_s = rs.details['V_Rd_max']
    vm_c = rc.details['V_Rd_max']
    ratio = vs_c / vs_s if vs_s > 0 else float('inf')

    print(f"{ct:<8.1f} | {vs_s:>11.1f} | {vs_c:>12.1f} | {ratio:>7.3f} | {vm_s:>13.1f} | {vm_c:>14.1f}")

print()
print("Note: V_Rd,s ratio reflects \u03bb\u2081 \u00d7 \u03bb\u2082 reduction for circular link efficiency.")
print("V_Rd,max ratio reflects the different b_w values used in K.")

Load: V_Ed=200.0 kN, M_Ed=150.0 kN·m, N_Ed=500.0 kN

cot(θ)   |  V_Rd,s std |  V_Rd,s circ |   Ratio |  V_Rd,max std |  V_Rd,max circ
--------------------------------------------------------------------------------
1.0      |       171.4 |        146.7 |   0.856 |         329.9 |          738.2
1.5      |       257.1 |        220.0 |   0.856 |         329.9 |          681.4
2.0      |       342.8 |        293.3 |   0.856 |         329.9 |          590.6
2.5      |       428.5 |        366.6 |   0.856 |         329.9 |          509.1

Note: V_Rd,s ratio reflects λ₁ × λ₂ reduction for circular link efficiency.
V_Rd,max ratio reflects the different b_w values used in K.


---
## 8. Multiple Load Cases: Full Comparison Table

A comprehensive table covering a range of realistic load combinations.

In [9]:
load_cases = [
    {"name": "Low V, no N",        "V": 100, "M": 80,  "N": 0},
    {"name": "Low V, high N",      "V": 100, "M": 50,  "N": 1200},
    {"name": "Moderate V",          "V": 200, "M": 150, "N": 500},
    {"name": "High V, low N",      "V": 350, "M": 200, "N": 200},
    {"name": "High V, high N",     "V": 350, "M": 100, "N": 1000},
    {"name": "Very high V",        "V": 500, "M": 100, "N": 500},
    {"name": "V with tension",     "V": 150, "M": 100, "N": -100},
    {"name": "Low V, high M",      "V": 80,  "M": 250, "N": 300},
]

header = (f"{'Load case':<22} | {'V_Ed':>5} {'M_Ed':>5} {'N_Ed':>6} | "
          f"{'cot\u03b8_s':>6} {'cot\u03b8_c':>6} | "
          f"{'VRds_s':>7} {'VRds_c':>7} | "
          f"{'VRdm_s':>7} {'VRdm_c':>7} | "
          f"{'Util_s':>7} {'Util_c':>7}")
print(header)
print("-" * len(header))

for lc_def in load_cases:
    lc = ShearLoadCase(V_Ed=lc_def["V"], M_Ed=lc_def["M"], N_Ed=lc_def["N"])
    rs = std.perform_check(load_case=lc)
    rc = circ.perform_shear_check(load_case=lc, force_cracked=True)

    ds = rs.details
    dc = rc.details

    print(f"{lc_def['name']:<22} | {lc_def['V']:>5} {lc_def['M']:>5} {lc_def['N']:>6} | "
          f"{ds['cot_theta']:>6.2f} {dc['cot_theta']:>6.2f} | "
          f"{ds['V_Rd_s']:>7.1f} {dc['V_Rd_s']:>7.1f} | "
          f"{ds['V_Rd_max']:>7.1f} {dc['V_Rd_max']:>7.1f} | "
          f"{rs.utilization:>6.1%} {rc.utilization:>6.1%}")

Load case              |  V_Ed  M_Ed   N_Ed | cotθ_s cotθ_c |  VRds_s  VRds_c |  VRdm_s  VRdm_c |  Util_s  Util_c
-----------------------------------------------------------------------------------------------------------------
Low V, no N            |   100    80      0 |   2.50   2.50 |   450.8   368.1 |   320.2   422.6 |  31.2%  27.2%
Low V, high N          |   100    50   1200 |   2.50   2.50 |   455.6   368.1 |   388.4   491.4 |  25.7%  27.2%
Moderate V             |   200   150    500 |   2.50   2.50 |   428.5   366.6 |   329.9   509.1 |  60.6%  54.5%
High V, low N          |   350   200    200 |   1.00   2.50 |   176.3   368.1 |   323.5   461.7 | 198.6%  95.1%
High V, high N         |   350   100   1000 |   1.49   2.50 |   271.0   368.1 |   377.9   482.0 | 129.1%  95.1%
Very high V            |   500   100    500 |   1.00   2.50 |   169.9   365.7 |   327.1   515.1 | 294.2% 136.7%
V with tension         |   150   100   -100 |   2.50   2.50 |   449.1   368.1 |   313.7   419.1 |  4

---
## 9. Uncracked vs Cracked: force_cracked Effect

`CircularSectionCheck` supports uncracked V_Rd,c (Eq.17, principal stress approach)
when `M_Ed < M_cr`. `ShearCheck` always uses the cracked empirical V_Rd,c.
The `force_cracked` parameter bypasses the M_cr check.

In [10]:
# Low-moment, high-axial case — likely uncracked
test_cases = [
    {"name": "Low M, high N",    "V": 150, "M": 30,  "N": 1000},
    {"name": "Med M, high N",    "V": 200, "M": 80,  "N": 800},
    {"name": "High M, low N",    "V": 200, "M": 200, "N": 100},
]

print(f"{'Load case':<22} | {'M_cr':>6} | {'Cracked?':>8} | "
      f"{'V_Rd auto':>10} {'V_Rd forced':>12} {'V_Rd std':>9} | "
      f"{'Check name (auto)':>40}")
print("-" * 125)

for tc in test_cases:
    lc = ShearLoadCase(V_Ed=tc["V"], M_Ed=tc["M"], N_Ed=tc["N"])

    # CircularSC auto (respects M_cr)
    r_auto = circ.perform_shear_check(load_case=lc)
    # CircularSC forced cracked
    r_forced = circ.perform_shear_check(load_case=lc, force_cracked=True)
    # Standard ShearCheck
    r_std = std.perform_check(load_case=lc)

    M_cr = r_auto.details.get('M_cr', None)
    is_cracked = r_auto.details['is_cracked']

    print(f"{tc['name']:<22} | {M_cr:>6.1f} | {str(is_cracked):>8} | "
          f"{r_auto.capacity:>10.1f} {r_forced.capacity:>12.1f} {r_std.capacity:>9.1f} | "
          f"{r_auto.check_name:>40}")

Load case              |   M_cr | Cracked? |  V_Rd auto  V_Rd forced  V_Rd std |                        Check name (auto)
-----------------------------------------------------------------------------------------------------------------------------
Low M, high N          |  189.7 |    False |      545.5        368.1     377.9 |        Circular shear (uncracked, Eq.17)
Med M, high N          |  174.6 |    False |      504.4        368.1     367.1 |        Circular shear (uncracked, Eq.17)
High M, low N          |  121.8 |     True |      368.1        368.1     320.8 |                  Circular shear (V_Rd_s)


---
## 10. b_w Breakdown: Polygon vs Circular Equivalent

The root cause of the differences is `b_w`. ShearCheck uses `calculate_section_breadth`
(minimum polygon chord width). For a circular polygon, this finds a narrow
chord near the top/bottom. Orr's circular `b_w = min(b_wc, b_wt)` is the chord
width at the compression and tension centroids — geometrically meaningful.

In [11]:
from materials.reinforced_concrete.code_checks.ec2_2004.shear_utils import calculate_section_breadth

b_w_polygon = calculate_section_breadth(section)

# Show circular b_w at different d/z values
print(f"Polygon minimum b_w: {b_w_polygon:.1f} mm")
print(f"Full diameter:       {D:.1f} mm")
print(f"Ratio polygon/D:     {b_w_polygon/D:.3f}")
print()

# For multiple load cases, show how circular b_w varies
print(f"{'N_Ed':>6} {'M_Ed':>6} | {'d':>6} {'z':>6} | {'b_wc':>6} {'b_wt':>6} {'b_w':>6} | {'b_w/D':>6}")
print("-" * 60)

for N in [0, 300, 600, 1000, 1500]:
    for M in [100, 200]:
        d = circ._shear_check.find_effective_depth(M, N)
        z_ec2, z_mech = circ._shear_check.find_lever_arm(M, N, d)
        z = z_mech if z_mech is not None else z_ec2
        b_w, b_wc, b_wt = circ.calculate_equivalent_web_width(d, z)
        print(f"{N:>6} {M:>6} | {d:>6.1f} {z:>6.1f} | {b_wc:>6.1f} {b_wt:>6.1f} {b_w:>6.1f} | {b_w/D:>6.3f}")

Polygon minimum b_w: 165.4 mm
Full diameter:       600.0 mm
Ratio polygon/D:     0.276

  N_Ed   M_Ed |      d      z |   b_wc   b_wt    b_w |  b_w/D
------------------------------------------------------------
     0    100 |  411.8  366.6 |  316.9  433.7  316.9 |  0.528
     0    200 |  411.8  364.8 |  322.4  433.7  322.4 |  0.537
   300    100 |  411.8  350.6 |  363.2  433.7  363.2 |  0.605
   300    200 |  411.8  355.4 |  350.1  433.7  350.1 |  0.584
   600    100 |  411.8  333.5 |  404.2  433.7  404.2 |  0.674
   600    200 |  411.8  348.4 |  368.9  433.7  368.9 |  0.615
  1000    100 |  411.8  370.6 |  303.4  433.7  303.4 |  0.506
  1000    200 |  411.8  343.5 |  381.3  433.7  381.3 |  0.635
  1500    100 |  411.8  370.6 |  303.4  433.7  303.4 |  0.506
  1500    200 |  411.8  328.8 |  414.4  433.7  414.4 |  0.691


---
## 11. Utilization Surface: V_Ed vs N_Ed

Heatmap comparing utilization across a grid of V_Ed and N_Ed values.

In [12]:
V_grid = np.arange(50, 450, 25)
N_grid = np.arange(0, 1600, 100)
M_fixed = 100

util_std_grid = np.zeros((len(N_grid), len(V_grid)))
util_circ_grid = np.zeros((len(N_grid), len(V_grid)))

for i, N in enumerate(N_grid):
    for j, V in enumerate(V_grid):
        lc = ShearLoadCase(V_Ed=float(V), M_Ed=M_fixed, N_Ed=float(N))
        rs = std.perform_check(load_case=lc)
        rc = circ.perform_shear_check(load_case=lc, force_cracked=True)
        util_std_grid[i, j] = min(rs.utilization, 3.0)  # Cap for visualisation
        util_circ_grid[i, j] = min(rc.utilization, 3.0)

fig = make_subplots(rows=1, cols=2, subplot_titles=("ShearCheck", "CircularSectionCheck"),
                    shared_yaxes=True)

fig.add_trace(go.Heatmap(z=util_std_grid, x=V_grid, y=N_grid, colorscale="RdYlGn_r",
                         zmin=0, zmax=2.0, colorbar=dict(title="Util", x=0.45)), row=1, col=1)
fig.add_trace(go.Heatmap(z=util_circ_grid, x=V_grid, y=N_grid, colorscale="RdYlGn_r",
                         zmin=0, zmax=2.0, colorbar=dict(title="Util", x=1.0)), row=1, col=2)

fig.update_xaxes(title_text="V_Ed (kN)")
fig.update_yaxes(title_text="N_Ed (kN)", row=1, col=1)
fig.update_layout(title=f"Utilization Heatmap (M_Ed={M_fixed} kN\u00b7m, force_cracked=True)", height=500)
fig.show()

---
## 12. Why the Differences Matter

Summary of key structural reasons for the differences.

In [13]:
lc = ShearLoadCase(V_Ed=250, M_Ed=150, N_Ed=500)
rs = std.perform_check(load_case=lc)
rc = circ.perform_shear_check(load_case=lc, force_cracked=True)

ds = rs.details
dc = rc.details

print("Key Differences Explained")
print("=" * 70)
print()
print("1. WEB WIDTH (b_w)")
print(f"   ShearCheck:    b_w = {ds['b_w']:.1f} mm (min polygon chord width)")
print(f"   CircularSC:    b_w = {dc['b_w']:.1f} mm (Orr Eq.10-13 equivalent)")
print(f"   Ratio:         {dc['b_w']/ds['b_w']:.2f}x")
print(f"   -> Polygon minimum is near the top/bottom of the circle where")
print(f"      chords are very narrow. Orr's method uses chords at the")
print(f"      compression and tension centroids, which is structurally correct.")
print()
print("2. STRUT ANGLE cot(\u03b8)")
print(f"   ShearCheck:    cot(\u03b8) = {ds['cot_theta']:.3f} (\u03b8 = {ds['theta_deg']:.1f}\u00b0)")
print(f"   CircularSC:    cot(\u03b8) = {dc['cot_theta']:.3f} (\u03b8 = {dc['theta_deg']:.1f}\u00b0)")
print(f"   -> Larger b_w gives larger K, so V_Rd,max is satisfied at a")
print(f"      flatter strut angle (higher cot\u03b8 = more efficient links).")
print()
print("3. LINK EFFICIENCY \u03bb\u2081")
print(f"   ShearCheck:    \u03bb\u2081 = 1.000 (assumes rectangular = full efficiency)")
print(f"   CircularSC:    \u03bb\u2081 = {dc.get('lambda_1', 'N/A'):.4f} (accounts for link curvature)")
print(f"   -> For circular links, only a component of the link force resists")
print(f"      vertical shear. \u03bb\u2081 ~ 0.85 is the typical reduction.")
print()
print("4. NET EFFECT ON V_Rd,s")
print(f"   ShearCheck:    V_Rd,s = {ds['V_Rd_s']:.1f} kN")
print(f"   CircularSC:    V_Rd,s = {dc['V_Rd_s']:.1f} kN")
print(f"   Ratio:         {dc['V_Rd_s']/ds['V_Rd_s']:.3f}")
print(f"   -> The higher cot\u03b8 (more efficient angle) partially compensates")
print(f"      for the \u03bb\u2081 reduction, so V_Rd,s values can be comparable.")
print()
print("5. V_Rd,max")
print(f"   ShearCheck:    V_Rd,max = {ds['V_Rd_max']:.1f} kN")
print(f"   CircularSC:    V_Rd,max = {dc['V_Rd_max']:.1f} kN")
print(f"   -> CircularSC has much higher V_Rd,max due to larger b_w.")
print(f"      ShearCheck may be governed by strut crushing prematurely")
print(f"      because its b_w is artificially narrow for circular sections.")

Key Differences Explained

1. WEB WIDTH (b_w)
   ShearCheck:    b_w = 165.4 mm (min polygon chord width)
   CircularSC:    b_w = 368.4 mm (Orr Eq.10-13 equivalent)
   Ratio:         2.23x
   -> Polygon minimum is near the top/bottom of the circle where
      chords are very narrow. Orr's method uses chords at the
      compression and tension centroids, which is structurally correct.

2. STRUT ANGLE cot(θ)
   ShearCheck:    cot(θ) = 2.181 (θ = 24.6°)
   CircularSC:    cot(θ) = 2.500 (θ = 21.8°)
   -> Larger b_w gives larger K, so V_Rd,max is satisfied at a
      flatter strut angle (higher cotθ = more efficient links).

3. LINK EFFICIENCY λ₁
   ShearCheck:    λ₁ = 1.000 (assumes rectangular = full efficiency)
   CircularSC:    λ₁ = 0.8556 (accounts for link curvature)
   -> For circular links, only a component of the link force resists
      vertical shear. λ₁ ~ 0.85 is the typical reduction.

4. NET EFFECT ON V_Rd,s
   ShearCheck:    V_Rd,s = 373.8 kN
   CircularSC:    V_Rd,s = 366.6 

---
## Summary

| Aspect | ShearCheck | CircularSectionCheck | Impact |
|--------|-----------|---------------------|--------|
| b_w | Min polygon chord (~165 mm) | Orr Eq.10-13 (~350-430 mm) | Dominates all differences |
| cot(θ) | Forced lower by small K | Higher (flatter strut) | More efficient link utilisation |
| λ₁ | 1.0 (rectangular assumption) | ~0.85 (circular link efficiency) | Reduces V_Rd,s by ~15% |
| V_Rd,s | Inflated by rectangular assumptions | Correct for circular geometry | Net effect depends on load |
| V_Rd,max | Unrealistically low for circular | Correct for circular geometry | ShearCheck may fail in strut crushing unnecessarily |
| V_Rd,c uncracked | Not available | Eq.17 principal stress | CircularSC can use higher capacity when uncracked |

**Conclusion:** `ShearCheck` applied to circular sections gives misleading results
because `calculate_section_breadth` finds the minimum polygon chord near the
section extremities, which is geometrically irrelevant for shear. The Orr (2012)
approach in `CircularSectionCheck` uses structurally meaningful web widths at
the compression and tension centroids, producing more accurate and typically
less conservative results.